<a href="https://colab.research.google.com/github/Shivakumarr18/NASA-ASRS-Pipelines/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
# 1. Read with multi-header
df = pd.read_csv("/content/ASRS_DBOnline.csv", header=[0, 1])

# 2. Flatten headers into a single row
df.columns = [f"{str(c).strip()}_{str(s).strip()}".replace(' ', '_')
              if not str(s).startswith('Unnamed')
              else str(c).strip().replace(' ', '_')
              for c, s in df.columns]

# 3. Clean up any weird double-underscores
df.columns = [c.replace('__', '_') for c in df.columns]

# 4. Save the "Clean Contract"
df.to_csv("ASRS_CLEAN_FLAT.csv", index=False)

In [9]:
# 1. Check the total number of columns
print(f"Total Columns: {df.shape[1]}")

# 2. List all column names to see the flattened results
print("\n--- Column List ---")
print(df.columns.tolist())

Total Columns: 125

--- Column List ---
['_ACN', 'Time_Date', 'Time_Local_Time_Of_Day', 'Place_Locale_Reference', 'Place_State_Reference', 'Place_Relative_Position.Angle.Radial', 'Place_Relative_Position.Distance.Nautical_Miles', 'Place_Altitude.AGL.Single_Value', 'Place_Altitude.MSL.Single_Value', 'Place_Latitude_/_Longitude_(UAS)', 'Environment_Flight_Conditions', 'Environment_Weather_Elements_/_Visibility', 'Environment_Work_Environment_Factor', 'Environment_Light', 'Environment_Ceiling', 'Environment_RVR.Single_Value', 'Aircraft_1_ATC_/_Advisory', 'Aircraft_1_Aircraft_Operator', 'Aircraft_1_Make_Model_Name', 'Aircraft_1_Aircraft_Zone', 'Aircraft_1_Crew_Size', 'Aircraft_1_Operating_Under_FAR_Part', 'Aircraft_1_Flight_Plan', 'Aircraft_1_Mission', 'Aircraft_1_Nav_In_Use', 'Aircraft_1_Flight_Phase', 'Aircraft_1_Route_In_Use', 'Aircraft_1_Airspace', 'Aircraft_1_Maintenance_Status.Maintenance_Deferred', 'Aircraft_1_Maintenance_Status.Records_Complete', 'Aircraft_1_Maintenance_Status.Rele

In [12]:
# 1. Load the file
df = pd.read_csv("ASRS_CLEAN_FLAT.csv")

# 2. Create a DataFrame of the column names
# We use index + 1 so the first column is '1' instead of '0'
column_list_df = pd.DataFrame({
    'Column_Number': range(1, len(df.columns) + 1),
    'Column_Name': df.columns
})

# 3. Save to Excel
output_file = "ASRS_Column_Audit.xlsx"
column_list_df.to_excel(output_file, index=False)

print(f"✅ Success! Your column list is ready: {output_file}")

✅ Success! Your column list is ready: ASRS_Column_Audit.xlsx


In [14]:
# 1. Total Rows and Columns
print(f"📊 DATASET AUDIT:")
print(f"- Total Rows (Incidents): {df.shape[0]}")
print(f"- Total Columns (Attributes): {df.shape[1]}")

# 2. Check for "Important" Missing Data
# We check if the critical 'Narrative' and 'ACN' are still there
critical_cols = [col for col in df.columns if 'ACN' in col or 'Narrative' in col]
print(f"\n✅ Critical Columns Verified: {critical_cols}")

📊 DATASET AUDIT:
- Total Rows (Incidents): 4501
- Total Columns (Attributes): 125

✅ Critical Columns Verified: ['_ACN', 'Report_1_Narrative', 'Report_2_Narrative']


In [16]:
# 1. Convert the null_percentage Series to a DataFrame
null_percentage_df = null_percentage.reset_index()
null_percentage_df.columns = ['Column_Name', 'Null_Percentage']

# 2. Filter out columns with 0% nulls (optional, but makes the report cleaner)
null_percentage_df = null_percentage_df[null_percentage_df['Null_Percentage'] > 0]

# 3. Sort by Null_Percentage in descending order
null_percentage_df = null_percentage_df.sort_values(by='Null_Percentage', ascending=False)

# 4. Save to Excel
output_null_file = "ASRS_Null_Value_Audit.xlsx"
null_percentage_df.to_excel(output_null_file, index=False)

print(f"✅ Success! Your null value audit is ready: {output_null_file}")

✅ Success! Your null value audit is ready: ASRS_Null_Value_Audit.xlsx


In [17]:
# --- PHASE 1: THE HIGH-INTEGRITY SILVER LAYER ---

# This list represents our 'Data Contract' approved for Risk Scoring
silver_contract = [
    # 1. PRIMARY IDENTITY
    '_ACN',

    # 2. TEMPORAL & SPATIAL (Context)
    'Time_Date', 'Time_Local_Time_Of_Day',
    'Place_State_Reference', 'Place_Altitude.MSL.Single_Value',

    # 3. ENVIRONMENT (External Risks)
    'Environment_Flight_Conditions', 'Environment_Light',

    # 4. THE ASSET (Aircraft 1)
    'Aircraft_1_Make_Model_Name', 'Aircraft_1_Aircraft_Operator',
    'Aircraft_1_Mission', 'Aircraft_1_Flight_Phase',

    # 5. THE SYSTEM (Component)
    'Component_1_Aircraft_Component', 'Component_1_Manufacturer', 'Component_1_Problem',

    # 6. THE FINDINGS (Assessments)
    'Events_Anomaly', 'Assessments_Primary_Problem', 'Assessments_Contributing_Factors_/_Situations',

    # 7. THE INTELLIGENCE (Narratives)
    'Report_1_Narrative', 'Report_1_Synopsis'
]

# Create the High-Integrity Silver DataFrame
df_silver = df[[col for col in silver_contract if col in df.columns]].copy()

# DATA HYGIENE: Convert Date to proper format (Critical for Time-Series Trust)
df_silver['Time_Date'] = pd.to_datetime(df_silver['Time_Date'], format='%Y%m', errors='coerce')

# AUDIT: Final Null Check for the VP
null_audit = df_silver.isnull().sum() / len(df_silver) * 100
print("📊 SILVER LAYER INTEGRITY AUDIT (Null %):")
print(null_audit[null_audit > 0]) # Only show columns that have issues

📊 SILVER LAYER INTEGRITY AUDIT (Null %):
_ACN                                              0.022217
Time_Date                                         0.022217
Time_Local_Time_Of_Day                           16.018663
Place_State_Reference                            11.197512
Place_Altitude.MSL.Single_Value                  53.099311
Environment_Flight_Conditions                    48.766941
Environment_Light                                65.518774
Aircraft_1_Make_Model_Name                        0.022217
Aircraft_1_Aircraft_Operator                      2.821595
Aircraft_1_Mission                                7.242835
Aircraft_1_Flight_Phase                           2.043990
Events_Anomaly                                    0.022217
Assessments_Primary_Problem                       0.022217
Assessments_Contributing_Factors_/_Situations     0.022217
Report_1_Narrative                                0.022217
Report_1_Synopsis                                 0.022217
dtype: float64


In [18]:
# Check how much 'Signal' is in the columns we are removing
discarded_range = df.iloc[:, 17:52] # Your Aircraft_1 range
print("📊 SIGNAL CHECK (Columns with more than 50% Data):")
high_signal = discarded_range.columns[discarded_range.notnull().mean() > 0.5]
print(high_signal.tolist())

📊 SIGNAL CHECK (Columns with more than 50% Data):
['Aircraft_1_Aircraft_Operator', 'Aircraft_1_Make_Model_Name', 'Aircraft_1_Crew_Size', 'Aircraft_1_Operating_Under_FAR_Part', 'Aircraft_1_Flight_Plan', 'Aircraft_1_Mission', 'Aircraft_1_Flight_Phase', 'Aircraft_1_Airspace', 'Component_Aircraft_Component']


In [19]:
# --- PHASE 1 FINAL: THE AUDITED SILVER LAYER ---

final_contract = [
    '_ACN', 'Time_Date',
    'Place_State_Reference', 'Place_Altitude.MSL.Single_Value',
    'Environment_Flight_Conditions',

    # Updated Aircraft Section with FAR Part
    'Aircraft_1_Make_Model_Name', 'Aircraft_1_Aircraft_Operator',
    'Aircraft_1_Mission', 'Aircraft_1_Flight_Phase',
    'Aircraft_1_Operating_Under_FAR_Part',

    'Component_Aircraft_Component', 'Component_1_Manufacturer', 'Component_1_Problem',
    'Events_Anomaly', 'Assessments_Primary_Problem', 'Assessments_Contributing_Factors_/_Situations',
    'Report_1_Narrative', 'Report_1_Synopsis'
]

# Create the final Silver DataFrame
df_silver = df[[col for col in final_contract if col in df.columns]].copy()

# Audit check
print(f"✅ Final Silver Layer Locked: {df_silver.shape[1]} Columns.")
print(f"Total Rows: {len(df_silver)}")

✅ Final Silver Layer Locked: 16 Columns.
Total Rows: 4501


In [21]:
# --- THE SYNCED SILVER LAYER ---

synced_contract = [
    '_ACN', 'Time_Date',
    'Place_State_Reference', 'Place_Altitude.MSL.Single_Value',
    'Environment_Flight_Conditions', 'Environment_Light',

    # Aircraft Section (Synced with your 'Signal Check' names)
    'Aircraft_1_Make_Model_Name', 'Aircraft_1_Aircraft_Operator',
    'Aircraft_1_Mission', 'Aircraft_1_Flight_Phase',
    'Aircraft_1_Operating_Under_FAR_Part',

    # Component Section (Fixed the naming here)
    'Component_Aircraft_Component', # Removed the '_1_' to match your file

    # Event & Narrative Section
    'Events_Anomaly', 'Assessments_Primary_Problem',
    'Assessments_Contributing_Factors_/_Situations',
    'Report_1_Narrative', 'Report_1_Synopsis'
]

# Create the final Silver DataFrame
df_silver = df[[col for col in synced_contract if col in df.columns]].copy()

print(f"✅ Synced Silver Layer Locked: {df_silver.shape[1]} Columns.")
display(df_silver.head(2))

✅ Synced Silver Layer Locked: 17 Columns.


,_ACN,Time_Date,Place_State_Reference,Place_Altitude.MSL.Single_Value,Environment_Flight_Conditions,Environment_Light,Aircraft_1_Make_Model_Name,Aircraft_1_Aircraft_Operator,Aircraft_1_Mission,Aircraft_1_Flight_Phase,Aircraft_1_Operating_Under_FAR_Part,Component_Aircraft_Component,Events_Anomaly,Assessments_Primary_Problem,Assessments_Contributing_Factors_/_Situations,Report_1_Narrative,Report_1_Synopsis
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2068539.0,202401.0,FL,7500.0,VMC,NaN,Amateur/Home Built/Experimental,Personal,Personal,Cruise,Part 91,NaN,ATC Issue All Types; Conflict NMAC; Deviation ...,Procedure,Chart Or Publication; Human Factors; Procedure...,I was enroute southeast bound on T208 VFR; in ...,Experimental aircraft pilot operating in ZMA a...


In [22]:
# See what we missed
missing = set(synced_contract) - set(df_silver.columns)
print(f"❌ Columns not found: {missing}")

❌ Columns not found: set()


In [23]:
# 1. Save the High-Integrity Subset
# This creates a 'Clean Contract' file for your Risk Engine
df_silver.to_csv("ASRS_SILVER_CLEAN.csv", index=False)

# 2. Final Audit Printout
print("💎 SILVER DATASET EXTRACTED")
print(f"Total Records: {df_silver.shape[0]}")
print(f"Total Features: {df_silver.shape[1]}")

# 3. Verify the file is in the Colab file system
import os
if os.path.exists("ASRS_SILVER_CLEAN.csv"):
    size_mb = os.path.getsize("ASRS_SILVER_CLEAN.csv") / (1024 * 1024)
    print(f"✅ File saved successfully! Size: {size_mb:.2f} MB")

# Look at the final product
display(df_silver.head())

💎 SILVER DATASET EXTRACTED
Total Records: 4501
Total Features: 17
✅ File saved successfully! Size: 7.89 MB


,_ACN,Time_Date,Place_State_Reference,Place_Altitude.MSL.Single_Value,Environment_Flight_Conditions,Environment_Light,Aircraft_1_Make_Model_Name,Aircraft_1_Aircraft_Operator,Aircraft_1_Mission,Aircraft_1_Flight_Phase,Aircraft_1_Operating_Under_FAR_Part,Component_Aircraft_Component,Events_Anomaly,Assessments_Primary_Problem,Assessments_Contributing_Factors_/_Situations,Report_1_Narrative,Report_1_Synopsis
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2068539.0,202401.0,FL,7500.0,VMC,NaN,Amateur/Home Built/Experimental,Personal,Personal,Cruise,Part 91,NaN,ATC Issue All Types; Conflict NMAC; Deviation ...,Procedure,Chart Or Publication; Human Factors; Procedure...,I was enroute southeast bound on T208 VFR; in ...,Experimental aircraft pilot operating in ZMA a...
2,2068551.0,202401.0,US,6500.0,VMC,Daylight,Skyhawk 172/Cutlass 172,FBO,Training,Cruise,Part 91,Throttle/Power Lever,Aircraft Equipment Problem Critical,Aircraft,Procedure; Human Factors; Aircraft,Departed ZZZ for PPL training. All controls no...,Cessna 172 Instructor reported a throttle malf...
3,2068838.0,202401.0,US,500.0,VMC,Daylight,PA-28 Cherokee/Archer/Dakota/Pillan/Warrior,FBO,Training,Landing,Part 91,NaN,ATC Issue All Types; Conflict NMAC,Procedure,Procedure,The aircraft I was flying was in a right downw...,GA flight crews reported same aircraft types a...
4,2068927.0,202401.0,FO,34000.0,VMC,Night,Widebody; Low Wing; 2 Turbojet Eng,Air Carrier,Passenger,Cruise,Part 121,GPS & Other Satellite Navigation,ATC Issue All Types; Aircraft Equipment Proble...,Aircraft,Aircraft; Company Policy; Human Factors; Proce...,I was on break as one of the relief pilots at ...,Air carrier flight crew reported GPS jamming a...
